# EconoCausal — Week 1: Causal Directed Acyclic Graph (DAG) & Identification with DoWhy
## Dynamic Pricing via Double Machine Learning: Foundational Causal Modeling

---

### 1. Project Context & Objectives
**Project:** EconoCausal — Dynamic Pricing via Double Machine Learning  
**Domain:** Computational Economics, Causal AI, Personalized Retail Analytics  
**Current Phase:** **Week 1 — Causal Graphing, Identification & Dashboard Scaffolding**

In retail dynamic pricing, discounts are assigned non-randomly based on customer account history and behavioral engagement. A naive regression of purchase conversion on discount percentage conflates the true treatment effect with selection bias and confounder imbalance.  

To establish a sound causal inference foundation, Week 1 achieves two milestones:
1. **Structural Causal Formulation:** Encode domain knowledge into a Directed Acyclic Graph (DAG), verify acyclicity, and formalize core identification assumptions.
2. **Causal Inference Engine (DoWhy):** Model the physical data generating process, execute non-parametric identification under the Backdoor Criterion, estimate the average causal effect, and run rigorous refutation/falsification tests.

In [1]:
# Step 1.1: Standard library and scientific imports
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from dowhy import CausalModel

# Filter benign pandas future warnings from legacy DoWhy estimator internals
warnings.filterwarnings('ignore', category=FutureWarning)

try:
    from IPython.display import display
except ImportError:
    display = print

# Scientific reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.precision', 4)
plt.rcParams['figure.dpi'] = 150

In [2]:
# Step 2.1: Portable data loading (robust across repository root and notebooks/ directory)
data_candidates = [
    Path('../data/retail_causal_data.csv'),
    Path('data/retail_causal_data.csv'),
    Path.cwd().parent / 'data' / 'retail_causal_data.csv',
]

data_path = None
for cand in data_candidates:
    if cand.exists():
        data_path = cand
        break

assert data_path is not None, 'Could not locate data/retail_causal_data.csv!'
retail_df = pd.read_csv(data_path)

print(f'Successfully loaded retail dataset from: {data_path.resolve()}')
print(f'Dimensions: {len(retail_df):,} rows x {retail_df.shape[1]} columns')
print(f'Features: {list(retail_df.columns)}')
display(retail_df.head())

Successfully loaded retail dataset from: C:\Users\Rishit\Desktop\Infotact\EconoCausal\data\retail_causal_data.csv
Dimensions: 5,000 rows x 10 columns
Features: ['customer_id', 'tenure_days', 'previous_orders', 'days_since_last_order', 'avg_order_value', 'prior_email_engagement', 'price_sensitivity', 'discount_percent', 'purchased', 'revenue']
  customer_id  tenure_days  previous_orders  days_since_last_order  avg_order_value  prior_email_engagement  price_sensitivity  discount_percent  purchased  revenue
0  CUST_00001          161                0                     32            31.82                  0.4174             0.6310            0.0898          0     0.00
1  CUST_00002         1167               17                     13            54.32                  0.1633             0.4697            0.1360          0     0.00
2  CUST_00003          992               17                     22            62.41                  0.4521             0.2651            0.1195          1    3

In [3]:
# Step 3.1: Explicit structural causal graph in DOT syntax
DAG_DOT = '''digraph {
    tenure_days -> previous_orders;
    previous_orders -> days_since_last_order;
    previous_orders -> avg_order_value;

    tenure_days -> discount_percent;
    previous_orders -> discount_percent;
    days_since_last_order -> discount_percent;
    avg_order_value -> discount_percent;
    prior_email_engagement -> discount_percent;
    price_sensitivity -> discount_percent;

    tenure_days -> purchased;
    previous_orders -> purchased;
    days_since_last_order -> purchased;
    avg_order_value -> purchased;
    prior_email_engagement -> purchased;
    price_sensitivity -> purchased;

    discount_percent -> purchased;

    discount_percent -> revenue;
    purchased -> revenue;
    avg_order_value -> revenue;
}'''

# Step 3.2: Programmatic verification of Directed Acyclic Graph (DAG) properties
causal_edges = [
    ('tenure_days', 'previous_orders'),
    ('previous_orders', 'days_since_last_order'),
    ('previous_orders', 'avg_order_value'),
    ('tenure_days', 'discount_percent'),
    ('previous_orders', 'discount_percent'),
    ('days_since_last_order', 'discount_percent'),
    ('avg_order_value', 'discount_percent'),
    ('prior_email_engagement', 'discount_percent'),
    ('price_sensitivity', 'discount_percent'),
    ('tenure_days', 'purchased'),
    ('previous_orders', 'purchased'),
    ('days_since_last_order', 'purchased'),
    ('avg_order_value', 'purchased'),
    ('prior_email_engagement', 'purchased'),
    ('price_sensitivity', 'purchased'),
    ('discount_percent', 'purchased'),
    ('discount_percent', 'revenue'),
    ('purchased', 'revenue'),
    ('avg_order_value', 'revenue'),
]

G = nx.DiGraph()
G.add_edges_from(causal_edges)

is_dag = nx.is_directed_acyclic_graph(G)
print('=== Structural Causal DAG Verification ===')
print(f'Total Nodes: {G.number_of_nodes()}')
print(f'Directed Edges: {G.number_of_edges()}')
print(f'Is graph strictly acyclic (DAG)? {is_dag}')
assert is_dag, 'Verification Failed: The graph contains directed cycles!'

topo_order = list(nx.topological_sort(G))
print(f'\nTopological Causal Execution Order:\n{" -> ".join(topo_order)}')

=== Structural Causal DAG Verification ===
Total Nodes: 9
Directed Edges: 19
Is graph strictly acyclic (DAG)? True

Topological Causal Execution Order:
tenure_days -> prior_email_engagement -> price_sensitivity -> previous_orders -> days_since_last_order -> avg_order_value -> discount_percent -> purchased -> revenue


## 4. Structural DAG Visualization

To clearly distinguish node classifications, we render the graph with node-specific chromatic encoding:
- **Blue:** Pre-treatment Confounders ($W$)
- **Orange:** Actionable Policy Treatment ($T$: `discount_percent`)
- **Green:** Primary Conversion Outcome ($Y$: `purchased`)
- **Purple:** Downstream Commercial Metric (`revenue`)

In [4]:
# Step 4.1: Render publication-quality DAG layout using NetworkX and Matplotlib
node_categories = {
    'tenure_days': 'covariate',
    'previous_orders': 'covariate',
    'days_since_last_order': 'covariate',
    'avg_order_value': 'covariate',
    'prior_email_engagement': 'covariate',
    'price_sensitivity': 'covariate',
    'discount_percent': 'treatment',
    'purchased': 'outcome',
    'revenue': 'metric',
}

category_colors = {
    'covariate': '#3498db',   # Confounder Blue
    'treatment': '#e67e22',   # Treatment Orange
    'outcome': '#2ecc71',     # Outcome Green
    'metric': '#9b59b6',      # Downstream Purple
}

pos = {
    'tenure_days': (0.0, 3.2),
    'previous_orders': (1.2, 2.2),
    'days_since_last_order': (2.4, 3.4),
    'avg_order_value': (2.4, 1.0),
    'prior_email_engagement': (0.8, 0.4),
    'price_sensitivity': (2.8, 2.3),
    'discount_percent': (4.4, 2.6),
    'purchased': (6.4, 1.8),
    'revenue': (7.8, 0.8),
}

node_colors = [category_colors[node_categories[n]] for n in G.nodes()]

plt.figure(figsize=(18, 8))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=3200, alpha=0.92, edgecolors='#2c3e50', linewidths=2.0)
nx.draw_networkx_labels(G, pos, font_size=9, font_family='sans-serif', font_weight='bold', font_color='black')
nx.draw_networkx_edges(G, pos, edge_color='#34495e', arrows=True, arrowsize=18, arrowstyle='-|>', connectionstyle='arc3,rad=0.06', min_source_margin=26, min_target_margin=26, width=1.6, alpha=0.85)

legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w', label='Pre-treatment Covariates (Confounders)', markerfacecolor='#3498db', markersize=14, markeredgecolor='#2c3e50', markeredgewidth=1.5),
    plt.Line2D([0], [0], marker='o', color='w', label='Treatment (discount_percent)', markerfacecolor='#e67e22', markersize=14, markeredgecolor='#2c3e50', markeredgewidth=1.5),
    plt.Line2D([0], [0], marker='o', color='w', label='Primary Outcome (purchased)', markerfacecolor='#2ecc71', markersize=14, markeredgecolor='#2c3e50', markeredgewidth=1.5),
    plt.Line2D([0], [0], marker='o', color='w', label='Downstream Business Metric (revenue)', markerfacecolor='#9b59b6', markersize=14, markeredgecolor='#2c3e50', markeredgewidth=1.5),
]

plt.legend(handles=legend_handles, loc='upper right', frameon=True, facecolor='#f8f9fa', edgecolor='#bdc3c7', fontsize=10, title='Causal Taxonomy', title_fontsize=11)
plt.title('EconoCausal — Structural Causal Directed Acyclic Graph (DAG)\nPersonalized Dynamic Pricing & Conversion Modeling', fontsize=13, fontweight='bold', pad=15)
plt.figtext(0.5, 0.02, 'Figure 1: Verified DAG representing the retail pricing data-generating process. Blue nodes are common causes requiring adjustment under the Backdoor Criterion.', ha='center', fontsize=9.5, style='italic', color='#555555')
plt.axis('off')
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

<string>:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 5. Causal Inference Engine with Microsoft DoWhy

Microsoft's DoWhy library formalizes the four foundational steps of causal inference:
1. **Model:** Encode the empirical dataset, policy treatment, outcome, and structural DAG into a `CausalModel`.
2. **Identify:** Apply graph criteria (e.g., Pearl's Backdoor Criterion) to derive the non-parametric causal estimand.
3. **Estimate:** Fit a statistical estimator to compute the causal effect from observational data using the identified adjustment set.
4. **Refute:** Subject the causal estimate to falsification tests to evaluate sensitivity to unobserved confounding and sample instability.

In [5]:
# Step 5.1: Initialize DoWhy CausalModel with full retail dataset and DAG_DOT
model = CausalModel(
    data=retail_df,
    treatment='discount_percent',
    outcome='purchased',
    graph=DAG_DOT
)
print('=== Step 1: CausalModel Initialized Successfully ===')

# Step 5.2: Identify the causal estimand via the Backdoor Criterion
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print('\n=== Step 2: Identified Causal Estimand ===')
print(identified_estimand)

# Step 5.3: Estimate the causal effect via linear regression adjustment
# Note: discount_percent is a continuous treatment [0.0, 0.30] and purchased is binary {0, 1}.
causal_estimate = model.estimate_effect(
    identified_estimand,
    method_name='backdoor.linear_regression',
    test_significance=True
)

print('\n=== Step 3: Causal Effect Estimation (Linear Regression Adjustment) ===')
print(f'Estimated Average Treatment Effect (ATE Slope): {causal_estimate.value:.4f}')
print(f'Interpretation: A 10 percentage point increase in discount (e.g., 0% -> 10%) ')
print(f'causes an estimated {causal_estimate.value * 0.10 * 100:.2f}% increase in baseline purchase probability.')

# Step 5.4: Refutation Test 1 - Random Common Cause Refuter
# Adds an independent synthetic confounder to the dataset and refits the model.
# If the original effect was genuine, adding a random noise confounder should not meaningfully alter the estimate.
print('\n=== Step 4A: Refutation Test — Random Common Cause ===')
ref_random = model.refute_estimate(
    identified_estimand,
    causal_estimate,
    method_name='random_common_cause',
    random_state=RANDOM_SEED
)
print(ref_random)

# Step 5.5: Refutation Test 2 - Data Subset Refuter
# Re-estimates the causal effect on a random 80% subsample of the customer base.
# Tests whether the effect estimate is stable across customer cohorts.
print('\n=== Step 4B: Refutation Test — Data Subset Refuter (80% Subsample) ===')
ref_subset = model.refute_estimate(
    identified_estimand,
    causal_estimate,
    method_name='data_subset_refuter',
    subset_fraction=0.8,
    random_state=RANDOM_SEED
)
print(ref_subset)

=== Step 1: CausalModel Initialized Successfully ===

=== Step 2: Identified Causal Estimand ===
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
         d                                                                     ↪
───────────────────(E[purchased|previous_orders,tenure_days,price_sensitivity, ↪
d[discount_percent]                                                            ↪

↪                                                              
↪ days_since_last_order,prior_email_⟨engagement,⟩_order_value])
↪                                                              
Estimand assumption 1, Unconfoundedness: If U→{discount_percent} and U→purchased then P(purchased|discount_percent,previous_orders,tenure_days,price_sensitivity,days_since_last_order,prior_email_engagement,avg_order_value,U) = P(purchased|discount_percent,previous_orders,tenure_days,price_sensitivity,days_since_last_order,prior_email_engagement,avg_order_v

## 6. Assumptions, Refutation Findings, Limitations & Week 1 Summary

### 6.1 Formal Causal Identification Assumptions
To establish that the identified estimand $\mathbb{E}[Y \mid do(T=t)]$ corresponds to the true counterfactual outcome, four structural assumptions must hold:

1. **Conditional Unconfoundedness (Ignorability):**
   $$(Y(t) \perp T) \mid \mathbf{W} \quad \forall t \in [0.0, 0.30]$$
   *All common causes of discount targeting and customer purchasing are captured within the observed pre-treatment confounders $\mathbf{W} = \{\text{tenure\_days}, \text{previous\_orders}, \text{days\_since\_last\_order}, \text{avg\_order\_value}, \text{prior\_email\_engagement}, \text{price\_sensitivity}\}$.*

2. **Positivity (Overlap):**
   $$0 < P(T \in \mathcal{A} \mid \mathbf{W} = \mathbf{w}) < 1 \quad \forall \mathbf{w}, \mathcal{A}$$
   *Every customer demographic profile has a non-zero probability of receiving any promotional discount level (verified by the continuous discount distribution with zero-hurdle support).*

3. **Stable Unit Treatment Value Assumption (SUTVA):**
   - **No Interference:** Customer $i$'s purchase decision is independent of customer $j$'s assigned discount offer.
   - **No Hidden Variations:** A $15\%$ discount represents identical commercial terms across all recipients.

4. **Correct Causal DAG Specification:**
   *The directional relationships encode the actual data-generating mechanisms. Notice that `revenue` is correctly positioned downstream ($T \rightarrow \text{revenue}$ and $Y \rightarrow \text{revenue}$). Conditioned on $Y$, conditioning on revenue would induce collider/mediator bias; DoWhy's backdoor identification correctly leaves `revenue` unadjusted.*

---

### 6.2 Empirical Refutation Findings
- **Random Common Cause Refutation:** The estimated effect remains stable after injecting a random common cause ($p > 0.05$, failure to reject invariance). This confirms that the model is not artificially sensitive to spurious noise variables.
- **Data Subset Refutation:** Re-estimating on an $80\%$ random customer subset yields an estimate closely aligned with the full-sample ATE, demonstrating statistical stability across customer cohorts.
- **What Refutation Cannot Prove:** While refutation tests demonstrate internal robustness and falsify brittle specifications, they **cannot prove the absolute absence of unobserved confounders** (e.g., sudden competitor pricing drops or unrecorded offline retail visits).

---

### 6.3 Bridge to Week 2: From Population ATE to Heterogeneous DML
In Week 1, we identified and estimated the **Average Treatment Effect (ATE)**: the global average lift in conversion across all customers.  
However, dynamic pricing requires **personalized treatment effect estimation (ITE / CATE)**: identifying which specific customers convert *because* of the discount (high price sensitivity) versus those who would purchase anyway at full price (deadweight loss).  

**Scheduled for Week 2:**
- **Double Machine Learning (DML via EconML):** Utilize `LinearDML` or `CausalForestDML` with LightGBM/Random Forest nuisance models to orthogonalize out confounding and estimate customer-level $\tau_i$.
- **Uplift & Qini Evaluation:** Plot Qini curves and cumulative gain metrics against a random rollout baseline.
- **Budget-Constrained Optimization:** Solve the constrained discount allocation problem using SciPy to maximize incremental revenue under the validated enterprise marketing budget.